In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from collections import Counter
import sys
import os
sys.path.append(os.path.abspath(".."))
from augmenter.custom_augment import CustomAugmenter

In [4]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:1


In [6]:
transform = transforms.Compose([
    CustomAugmenter(),                   
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  
])

dataset_path = "../dataset"
full_dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
class_names = full_dataset.classes
num_classes = len(class_names)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

def count_classes(dataset, full_dataset):
    indices = dataset.indices if isinstance(dataset, torch.utils.data.Subset) else range(len(dataset))
    labels = [full_dataset[i][1] for i in indices]
    return Counter(labels)

train_class_counts = count_classes(train_dataset, full_dataset)
val_class_counts = count_classes(val_dataset, full_dataset)

print("\nClass distribution in training set:")
for cls_idx, count in train_class_counts.items():
    print(f"  {class_names[cls_idx]}: {count} samples")

print("\nClass distribution in validation set:")
for cls_idx, count in val_class_counts.items():
    print(f"  {class_names[cls_idx]}: {count} samples")


train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)



Class distribution in training set:
  single_tone: 8052 samples
  wideband: 7971 samples
  benign: 8164 samples
  pulse: 7985 samples

Class distribution in validation set:
  benign: 2036 samples
  single_tone: 1951 samples
  wideband: 2035 samples
  pulse: 2022 samples


In [ ]:
model = models.mobilenet_v2(pretrained=True)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Move to device
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / train_size
    train_acc = 100.0 * correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss = val_loss / val_size
    val_acc = 100.0 * val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

Epoch [1/10] Train Loss: 0.2000, Train Acc: 92.17% Val Loss: 0.1811, Val Acc: 93.91%
Epoch [2/10] Train Loss: 0.0668, Train Acc: 97.54% Val Loss: 0.0734, Val Acc: 97.33%
Epoch [3/10] Train Loss: 0.0495, Train Acc: 98.26% Val Loss: 0.0470, Val Acc: 98.21%
Epoch [4/10] Train Loss: 0.0359, Train Acc: 98.68% Val Loss: 0.0170, Val Acc: 99.42%
Epoch [5/10] Train Loss: 0.0298, Train Acc: 98.95% Val Loss: 0.0174, Val Acc: 99.34%
